# 1. Model fitting with LOCO and LOYO


In [ ]:
import _bootstrap  # noqa: F401  — puts src/ on sys.path (see src/analysis/README.md)
import utils
import params

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error
import os
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso
from statsmodels import api as sm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from pathlib import Path
# Initialize the scaler
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings("ignore")

from matplotlib import pyplot
pyplot.rcParams['figure.dpi'] = 300
pyplot.rcParams['savefig.dpi'] = 600

In [ ]:
# Initialize empty DataFrames to store results
country_specific_df = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', 'country', 'pred', 'true', 'train_size'])
year_specific_df = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', 'year', 'pred', 'true', 'train_size'])

# Define a list of CIS countries
CIS_countries = params.CIS_countries


def deleting_itu_func(data,outcome_var,indicator,CIS_control=False):
    """
    Function to filter and preprocess data by removing ITU-related entries.
    
    Args:
        data (pd.DataFrame): Input dataset.
        outcome_var (str): Target variable for the model.
        indicator (str): Indicator variable (e.g., 'internet', 'mobile').
        CIS_control (bool): Whether to apply specific rules for CIS countries.

    Returns:
        pd.DataFrame: Processed dataset with ITU-related entries removed.

    """
    data = data.loc[~data['iso3'].isin(params.sanction_countries)]
    if CIS_control:
        for iso3 in data.iso3.unique():
            if iso3 in CIS_countries:
                if ('itu' in data.loc[data['iso3']==iso3,f'{indicator}_survey_type'].values) & (data.loc[data['iso3']==iso3,f'{indicator}_survey_type'].nunique()==1):
                        continue # keep all values if only itu surveys available
                    
                else:
                    data.loc[data['iso3']==iso3,f'{outcome_var}']=[x if y!='itu' else None for x,y in zip(data.loc[data['iso3']==iso3,f'{outcome_var}'],data.loc[data['iso3']==iso3,f'{indicator}_survey_type'])]
            
            elif data.loc[data['iso3']==iso3,'conti'].values[0] in ['Europe','North America','South America']:
                if ('itu' in data.loc[data['iso3']==iso3,f'{indicator}_survey_type'].values) & (data.loc[data['iso3']==iso3,f'{indicator}_survey_type'].nunique()==1):
                    
                    #survey_values = [x if y>0.85 else None for x,y in zip(data.loc[data['iso3']==iso3,f'{outcome_var}'],data.loc[data['iso3']==iso3,f'internet_wom'])]
                    survey_values = [x if y>0.85 else None for x,y in zip(data.loc[data['iso3']==iso3,f'{outcome_var}'],data.loc[data['iso3']==iso3,f'{indicator}_wom'])]
                    data.loc[data['iso3']==iso3,f'{outcome_var}']=  survey_values
                    #print(data.loc[data['iso3']==iso3,f'{outcome_var}'].values),
                else:
                    data.loc[data['iso3']==iso3,f'{outcome_var}']=[x if y!='itu' else None for x,y in zip(data.loc[data['iso3']==iso3,f'{outcome_var}'],data.loc[data['iso3']==iso3,f'{indicator}_survey_type'])]
            else:
                data.loc[data['iso3']==iso3,f'{outcome_var}']=[x if y!='itu' else None for x,y in zip(data.loc[data['iso3']==iso3,f'{outcome_var}'],data.loc[data['iso3']==iso3,f'{indicator}_survey_type'])]

    else:
        data = data.loc[data[f'{indicator}_survey_type']!='itu']
    return data



def save_scaler(scaler, file_path):
    joblib.dump(scaler, file_path)

def save_model(model, file_path):
    joblib.dump(model, file_path)


def load_model(file_path):
    return joblib.load(file_path) if os.path.exists(file_path) else None


# Function to preprocess data
def preprocess_data(data):
    non_numeric_cols = data.select_dtypes(exclude=['number']).columns
    if len(non_numeric_cols) > 0:
        data = pd.get_dummies(data, columns=non_numeric_cols)
    return data

# Main function with model selection
def evaluate_model(indicator, outcome_var, dataset, model_type, model_spec, regression_method, alpha=0.5, country_specific_df=country_specific_df, year_specific_df=year_specific_df):
    """
    Evaluate a regression model using the specified method (OLS or Lasso).

    Args:
        indicator (str): Indicator variable (e.g., 'internet', 'mobile').
        outcome_var (str): Target variable for the model.
        dataset (pd.DataFrame): Input dataset.
        model_type (str): Type of model (e.g., 'combined', 'online').
        model_spec (list): List of feature columns for the model.
        regression_method (str): Regression method ('OLS' or 'Lasso').
        alpha (float): Regularization parameter for Lasso regression.
        country_specific_df (pd.DataFrame): DataFrame to store country-specific results.
        year_specific_df (pd.DataFrame): DataFrame to store year-specific results.

    Returns:
        tuple: Updated year-specific and country-specific DataFrames, and results DataFrame.
    """
    data = dataset.copy()
    
    # Drop rows where the target variable is missing
    if deleting_itu:
        data = deleting_itu_func(data, outcome_var, indicator, True if 'CIS' in model_type else False)
        print(f'{outcome_var}: {data[outcome_var].notnull().sum()} samples left for modelling')
    data = data.dropna(subset=[outcome_var])
    
    # Preprocess the data
    X = preprocess_data(data[model_spec])
    y = data[outcome_var].astype(float)
    
    # Define file paths for saving the model
    model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl' if deleting_itu else f'{indicator}_{model_type}_{outcome_var.replace(f"{indicator}_", "")}_{outcome_var}_full_model.pkl'
    model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
    
    # Train the model based on the specified regression method
    if regression_method == 'OLS':
        X_const = sm.add_constant(X)  # Add constant term for intercept
        model = sm.OLS(y, X_const).fit()
        y_pred_full = model.predict(X_const)
        r2_full_sample = model.rsquared
        r2_adj = model.rsquared_adj
    elif regression_method == 'Lasso':
        model = Lasso(alpha=alpha)
        model.fit(X, y)
        y_pred_full = model.predict(X)
        r2_full_sample = r2_score(y, y_pred_full)
        r2_adj = 1 - (1 - r2_full_sample) * (len(y) - 1) / (len(y) - X.shape[1] - 1)
    else:
        raise ValueError("Unsupported regression method. Choose 'OLS' or 'Lasso'.")
    
    # Save the trained model
    save_model(model, model_filepath)


    # Variable importance/selection (only for Lasso)
    selected_vars = []
    if regression_method == 'Lasso':
        selected_vars = [var for var, coef in zip(X.columns, model.coef_) if coef != 0]
    else:
        selected_vars = model_spec

    # Leave-one-out evaluation

    def leave_one_column_out(column):
        """
        Perform leave-one-out evaluation for a specific column.

        Args:
            column (str): Column to leave out.

        Returns:
            tuple: DataFrame with predictions and R² score.
        """
        recorder = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', column, 'pred', 'true', 'train_size'])
        for unique_value in data[column].unique():

            if regression_method == 'OLS':
                X_train_loo = X_const[data[column] != unique_value]
                y_train_loo = y[data[column] != unique_value]
                X_test_loo = X_const[data[column] == unique_value]
                y_test_loo = y[data[column] == unique_value]

                model = sm.OLS(y_train_loo, X_train_loo).fit()
                y_pred_loo = model.predict(X_test_loo).values


            else:
                X_train_loo = X[data[column] != unique_value]
                y_train_loo = y[data[column] != unique_value]
                X_test_loo = X[data[column] == unique_value]
                y_test_loo = y[data[column] == unique_value]

                model.fit(X_train_loo, y_train_loo)
                y_pred_loo = model.predict(X_test_loo)

            for i in range(len(y_pred_loo)):

                recorder.loc[len(recorder)] = [f'{indicator} {model_type}', outcome_var, model_type, unique_value, y_pred_loo[i], y_test_loo.values[i], len(y_train_loo)]

        r2 = r2_score(recorder['true'], recorder['pred'])

        return recorder, r2
    
    # Perform leave-one-out evaluations for 'iso3' and 'year'
    r2s = {}
    for leave_column in ['iso3', 'year']:
        temp, r2s[leave_column] = leave_one_column_out(leave_column)
        if leave_column == 'iso3':
            country_specific_df = pd.concat([country_specific_df, temp], axis=0)
        elif leave_column == 'year':
            year_specific_df = pd.concat([year_specific_df, temp], axis=0)

    # Append results
    results_df.loc[len(results_df)] = {
        "model_name": f'{indicator} {model_type}',
        'method': regression_method,
        'outcome_var': outcome_var,
        "model_type": model_type,
        'best_r2': r2_full_sample,
        'adj_r2': r2_adj,
        'loco_r2': r2s['iso3'],
        'loyo_r2': r2s['year'],
        'selected_var': ', '.join(selected_vars)
    }

    return year_specific_df, country_specific_df, results_df


In [ ]:
#
for indicator in ['internet', 'mobile']:
    data_path = params.dropbox_data_path / f"combined_data/updated_ground_truth_and_fb/{indicator}"
    data = pd.read_csv(data_path / f"combined_multiple_years_no_missing_keep_countries_fb_aligned.csv")
    data = deleting_itu_func(data, f'{indicator}_ggi', indicator,True)
    data.dropna(subset=[f'{indicator}_ggi'], inplace=True)  # Ensure no NaN in target variable
    # data,outcome_var,indicator,CIS_control=False
    print(f'{indicator} data shape: {data.shape}, unique countries: {data.iso3.nunique()}')
    print(data[f'{indicator}_survey_type'].value_counts())

## Load data and fit model 

In [ ]:
deleting_itu = True 
with_year = True
regression_method = 'OLS'

# define zones
fb_cols =["fb_18_999_men","fb_18_999_wom","fb_18_999_r"]
#bg_cols = params.bg_cols
bg_cols = ['hdi','gdi','gdp_pcap','year']
model_specs = {'online_with_CIS': fb_cols, 
                'offline_with_CIS': bg_cols, 
                #'combined': fb_cols + bg_cols,
                #'combined_top_shaps':['gggi_ggi','se_r','hdi','gdp_pcap','wdi_fertility','year']+fb_cols,
                #'combined_top_shaps_no_fertility':['gggi_ggi','se_r','hdi','gdp_pcap','year']+fb_cols,
                "combined_with_CIS":fb_cols + bg_cols}
                #"combined_top_shaps_from_mobile":['wdi_litrad','gni_pc','hdi','year','ineq_le','wdi_acel','gii']+fb_cols}

# Define a list of CIS countries
CIS_countries = ['BEL', 'KAZ', 'ARM', "KGZ", 'MDA', "AZE", 'TKJ', 'UZB', 'TKM']



In [ ]:
# Initialize an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['model_name', 'method', 'outcome_var', 'model_type', 'best_r2','adj_r2', 'loco_r2', 'loyo_r2', 'selected_var'])
country_specific_df = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', 'country', 'pred', 'true', 'train_size'])
year_specific_df = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', 'year', 'pred', 'true', 'train_size'])

from pathlib import Path
df_outcomes = pd.read_csv(Path("/Users/valler/Dropbox/dgg_research/national/data_refresh/new_national_pipeline_files/files/new_groundtruth_national_data.csv"))
df_outcomes = df_outcomes.loc[df_outcomes['survey_year']>=2015]

df_outcomes=df_outcomes.pivot_table(index=['gid_0','source','survey_year'],columns="outcome",values='observed').reset_index()



for indicator in ['internet','mobile']:
    
    data_path = params.dropbox_data_path / f"combined_data/updated_ground_truth_and_fb/{indicator}"
    data = pd.read_csv(data_path / f"combined_multiple_years_no_missing_keep_countries_fb_aligned.csv")
    data['year']=data[f'{indicator}_year']-2015
    # deleting ITU 
    if indicator=='mobile':
        data = data.merge(df_outcomes[['gid_0','source','survey_year','internet_women']],left_on=['iso3',f'{indicator}_year',f'{indicator}_survey_type'],right_on=['gid_0','survey_year','source'],how='left')
        data.rename(columns={'internet_women':'internet_wom'},inplace=True)
       

    for model_type in model_specs.keys():
        for outcome_var in [f'{indicator}_ggi', f'{indicator}_wom', f'{indicator}_men']:
            year_specific_df, country_specific_df, results_df = evaluate_model(indicator,outcome_var, data, model_type,model_specs[model_type], regression_method, 0.1, country_specific_df, year_specific_df)
    


In [ ]:
# save the results
country_specific_df.to_csv(params.dropbox_data_path.parent/f'results/logs/ols{"_no_ITU" if deleting_itu else ""}{"_with_year" if with_year else ""}_loco_results.csv', index=False)
year_specific_df.to_csv(params.dropbox_data_path.parent/f'results/logs/ols{"_no_ITU" if deleting_itu else ""}{"_with_year" if with_year else ""}_loyo_results.csv', index=False)
results_df.to_csv(params.dropbox_data_path.parent/f'results/logs/ols{"_no_ITU" if deleting_itu else ""}{"_with_year" if with_year else ""}_results_summary.csv', index=False)

## results compare

In [ ]:
results_df=pd.read_csv(params.dropbox_data_path.parent/f'results/logs/ols{"_no_ITU" if deleting_itu else ""}{"_with_year" if with_year else ""}_results_summary.csv')
results_df_CIS=results_df.loc[results_df['model_type'].str.contains('CIS')]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from pathlib import Path

# --- Prep ---
df = results_df_CIS.copy()
df['model_type'] = df['model_type'].str.replace('_with_CIS', '', regex=False)

# Format labels for x-axis
df["formatted_label"] = (
    df["outcome_var"]
    .str.capitalize()
    .str.replace("_", "\n", regex=False)
    .str.replace("men", "Men level", regex=False)
    .str.replace("wom", "Women level", regex=False)
    .str.replace("ggi", "GGI", regex=False)
)

# Split into internet and mobile outcome sets
df_internet = df[df["outcome_var"].str.contains("internet", case=False)]
df_mobile = df[df["outcome_var"].str.contains("mobile", case=False)]

# Palette for model_type
model_type_palette = {
    "combined": "#3a5b7e",
    "online": "#f0935d",
    "offline": "#74a4bc"
}

# --- Plot ---
fig, axs = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

for ax, subset, title in zip(
    axs,
    [df_internet, df_mobile],
    ["Internet-related outcomes", "Mobile-related outcomes"]
):
    sns.barplot(
        data=subset,
        x="formatted_label",
        y="loco_r2",
        hue="model_type",
        palette=model_type_palette,
        ax=ax,
        saturation=1
    )

    # Title & labels
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel("Leave-one-country-out  R²" if "Internet" in title else "")
    if "Mobile" in title:
        ax.set_yticklabels([])  # hide y-ticks on right plot
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center')

    # Black edges on bars
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(1)

    # Add labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', label_type='edge', fontsize=6)

    # Remove clutter
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0)

# --- Legend ---
legend_elements = [
    Patch(facecolor=model_type_palette[m], edgecolor='black', linewidth=1.2, label=m.capitalize())
    for m in ['online', 'offline', 'combined']
]

#axs[0].legend_.remove()
axs[0].legend_.remove()
axs[1].legend(title='Model Type', handles=legend_elements, loc='upper right')

# --- after plotting ---
plt.tight_layout()

# Add a more separated suptitle
fig.suptitle(
    'a. Adult Internet and Mobile Outcomes',
    fontsize=12,
    fontweight='bold',
    y=1.05,x=0.25# push it higher above the subplot titles
)

# Add extra top margin so suptitle doesn't overlap with subplots
plt.subplots_adjust(top=0.9)

plt.savefig(
    Path.cwd().parent / 'figure/ols_loco_r2_by_outcome_var_grouped.pdf',
    bbox_inches='tight'
)


## print out model coefficients

In [ ]:
from statsmodels.iolib.summary2 import summary_col
pd.options.display.max_rows = 4000

# load statistics 
all_models = {}

for indicator in ['internet', 'mobile']:
    all_models[indicator] = []
    for outcome_var in [f'{indicator}_ggi', f'{indicator}_wom', f'{indicator}_men']:
        for model_type in ['combined_with_CIS']:
            model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl'
            model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
            model = load_model(model_filepath)
            all_models[indicator].append(model)

In [ ]:
# internet results in latex format
print(summary_col(all_models['internet']+all_models['mobile'],stars=True,float_format='%0.3f').as_latex())

## pred vs true 

In [ ]:
df_all_res = pd.DataFrame()

for indicator in ['internet', 'mobile']:
    data_path = params.dropbox_data_path / f"combined_data/updated_ground_truth_and_fb/{indicator}"
    data = pd.read_csv(data_path / f"combined_multiple_years_no_missing_keep_countries_fb_aligned.csv")
    data['year'] = data[f'{indicator}_year'] - 2015
    for outcome_var in [f'{indicator}_ggi', f'{indicator}_wom', f'{indicator}_men']:
        
        data[outcome_var] = round(data[outcome_var], 5)
        for model_type in ['combined_with_CIS']:
            
            model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl'
            model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
            model = load_model(model_filepath)
            
            predicted_values = model.predict(model.model.exog)
            df_res = pd.DataFrame({
                'year': model.model.data.orig_exog['year'],
                'pred': predicted_values,
                'outcome_var': outcome_var,
                'indicator': indicator,
                'model_type': model_type,
                'true': model.model.endog
            })
            df_res['true'] = df_res['true'].apply(lambda x: round(x, 5))
            df_res = df_res.merge(data[['iso3', outcome_var,'year']], left_on=['true','year'],right_on=[outcome_var,'year'], how='left')
            # for countries with multiple years, take the latest year
            df_res = df_res.sort_values(by=['iso3', 'year']).drop_duplicates(subset=['iso3', 'pred'], keep='last')
            df_all_res = pd.concat([df_all_res, df_res], ignore_index=True)
    

In [ ]:
df_all_res = utils.mark_regions(df_all_res)
df_all_res['pred'] = df_all_res['pred'].clip(0, 1)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

hue_column = 'conti'  # column for color coding
outcome_name_dict = {"ggi":"Female-Male Ratio","men":"Men Level","wom":"Women Level"}
# ---------------- Prep ----------------

region_colors = {
    "Asia":"#e76254",  # Coral red
    "North America":"#7db782",  # Yellow-orange
    "Africa":"#f1975a",  # Light beige
    "South America":"#72c0c5",  # Pale cyan
    "Oceania":"#cf8dc5",  # Light blue
    "Europe":"#5f799a",  # Medium blue
    "Global":"#101c2c"   # Dark blue
}

# Split "outcome_var" -> indicator + outcome_core (e.g., "internet_ggi")
parts = df_all_res['outcome_var'].str.extract(r'^(internet|mobile)_(ggi|wom|men)$')
df_plot = df_all_res.copy()
df_plot['indicator'] = parts[0]
df_plot['outcome_core'] = parts[1]
df_plot = df_plot.dropna(subset=['indicator', 'outcome_core'])

# ---------------- Subplot Setup ----------------
fig, axes = plt.subplots(2, 3, figsize=(11, 8))
outcome_cores = ['ggi', 'wom', 'men']
indicators = ['internet', 'mobile']

for i, indicator in enumerate(indicators):
    for j, outcome in enumerate(outcome_cores):
        ax = axes[i, j]
        data = df_plot[(df_plot['indicator'] == indicator) & (df_plot['outcome_core'] == outcome)]

        sns.scatterplot(
            data=data,
            x='true',
            y='pred',
            hue=hue_column,
            palette=region_colors,
            s=25,
            alpha=1,
            ax=ax,
            legend=False if i + j > 0 else True  # show legend only once
        )

        ax.plot([-0.05, 1.05], [-0.05, 1.05], color='grey', linestyle='--', linewidth=1)
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_aspect('equal', adjustable='box')


        ax.set_xlabel("True values")
        ax.set_ylabel("Predicted values")


        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax.set_title(f"{indicator.capitalize()} {outcome_name_dict.get(outcome, outcome)}", fontweight='bold')

# remove legend from first subplot
axes[0, 0].legend_.remove()

# Adjust layout and legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title='Continent', loc='lower center', ncol=6, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])

# save or show
plt.savefig(Path.cwd().parent / "figure/adult_pred_vs_true.pdf", bbox_inches='tight')


# 2. Model prediction errors 

In [ ]:
def leave_one_column_out_for_error_estimation(column, data, outcome_var, indicator, X, y):
    """
    Perform leave-one-out validation for a specific column.

    Args:
        column (str): Column to leave out (e.g., 'country', 'year').
        data (pd.DataFrame): Input dataset.
        outcome_var (str): Target variable for the model.
        indicator (str): Indicator type (e.g., 'internet', 'mobile').
        X (pd.DataFrame): Feature matrix.
        y (pd.Series or np.ndarray): Target variable values.

    Returns:
        tuple: A DataFrame with predictions and true values, R² score, and mean absolute error.
    """
    recorder = pd.DataFrame(columns=["name", 'outcome_var', 'model_type', column, 'year', 'pred', 'true'])
    
    for unique_value in data[column].unique():
        # Split data into training and testing sets
        X_test_loo = X[data[column] == unique_value]
        y_test_loo = y[data[column] == unique_value]
        
        X_train_loo = X[data[column] != unique_value]
        y_train_loo = y[data[column] != unique_value].tolist()
        
        years = data.loc[X_test_loo.index, 'year'].tolist()
        
        # Train the model using OLS regression
        model = sm.OLS(y_train_loo, X_train_loo).fit()
        
        # Predict on the excluded data
        y_pred_loo = model.predict(X_test_loo).values

        # Record predictions and true values
        for i in range(len(y_pred_loo)):
            recorder.loc[len(recorder)] = [
                f'{indicator} {model_type}', outcome_var, model_type, unique_value, years[i], y_pred_loo[i], y_test_loo.values[i]
            ]

    # Calculate performance metrics
    r2 = r2_score(recorder['true'], recorder['pred'])
    mean_abs = np.abs(recorder['true'] - recorder['pred']).mean()
    
    return recorder, r2, mean_abs

In [ ]:
from scipy.optimize import nnls
from sklearn.metrics import  r2_score

model_folder = 'OLS'
leave_column = 'country'

model_type='combined_with_CIS'
model_spec = model_specs[model_type]

coef = pd.DataFrame(columns =['model','r2','mean_abs_diff']+list(model_spec))

df_errors = pd.DataFrame()

dataset = "combined_multiple_years_no_missing_keep_countries_fb_aligned.csv"
for indicator in ['internet', 'mobile']:
    data_path = params.dropbox_data_path / f"combined_data/{indicator}"
    # only combined model
   
    for outcome_var in [f'{indicator}_ggi', f'{indicator}_wom', f'{indicator}_men']:
        
         # read and preprocess the previous model data 
        data = pd.read_csv(data_path / dataset)
        data.rename(columns={f'{indicator}_year':'year'}, inplace=True)

        #data = dataset.copy()
    # Drop rows where the target variable is missing
        if deleting_itu:
            data=deleting_itu_func(data,outcome_var,indicator,True if 'CIS' in model_type else False)
            print(f'{outcome_var}:{data[outcome_var].notnull().sum()} samples left for modelling')
        # data = data.dropna(subset=[outcome_var])
  
        data = data.dropna(subset=model_spec) # important! remove countries with missing values in FB features 
        data = data.dropna(subset=[outcome_var])
    
        # load full model to get the specs 
        model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl' 
        model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
        
        # Try to load the model if it exists
        rf_loo = load_model(model_filepath)
        model_spec = model_specs[model_type]

        # preprocess data 
        X = data[model_spec]
        y = data[outcome_var]
        # store the true/pred values 
        df_res, r2, abs_mean_error = leave_one_column_out_for_error_estimation(leave_column,data,outcome_var,indicator,X,y)
        
        data = data.merge(df_res[['country','pred','true','year']],on=['country','year'],how='left')
        data.rename(columns={'pred':'predicted'},inplace=True)
         
        # calculate the absolute difference
        data['abs_diff'] = np.abs(data[outcome_var] - data['predicted'])
        data = data[['iso3', 'country', 'year',outcome_var,'predicted','abs_diff']+list(model_spec)]
        
        data['year']=data['year']-2015
        # now fit the model 
        model_name = outcome_var
        # Perform non-negative linear regression
        coefficients, residual = nnls(data[list(model_spec)+['year']], data['abs_diff'])
        residual_pred = data[list(model_spec)+['year']] @ coefficients 
        
        data['predicted_error'] = residual_pred
        residual_r2= r2_score(data['abs_diff'],residual_pred)
        data = data[['iso3', 'country', 'year',outcome_var,'predicted','abs_diff','predicted_error']]
         
        #df_errors = pd.concat([df_errors,data],ignore_index=True)
        abs_diff = data['abs_diff'].mean()
        print(round(abs_mean_error,3)==round(abs_diff,3))
         
        
        new_row = {'model': model_name,'r2': residual_r2,'mean_abs_diff': abs_diff}
        coef_dict = dict(zip(list(model_spec)+['year'], coefficients)) 
        new_row.update(coef_dict)
         
        new_row = pd.DataFrame(new_row,index=range(len(new_row))).drop_duplicates()
        coef = pd.concat([coef,new_row],ignore_index=True)
        #coef.reset_index(drop=True,inplace=True)

# coef.to_csv(params.dropbox_data_path.parent / f'results/ols_predicted_by_year/{model_type}{f"_{leave_column}" if len(leave_column)>0 else "" }_model_error_estimation_betas.csv',index=False)

In [ ]:
# save the coefficients and results
coef.to_csv(params.dropbox_data_path.parent / f'results/ols_predicted_by_year/{model_type}{f"_{leave_column}" if len(leave_column)>0 else "" }_model_error_estimation_betas.csv',index=False)

# 3. Generate yearly predicted results 

In [ ]:
def full_model_predictions(
    indicator,
    new_data,
    model_type,
    model_spec,
    outcome_var,
    align_year,
    model_coef
):
    """
    Generate predictions for new data using a trained model and estimate prediction error.

    Args:
        indicator (str): Indicator type ('internet' or 'mobile').
        new_data (pd.DataFrame): DataFrame containing new data for prediction.
        model_type (str): Model type (e.g., 'combined_with_CIS').
        model_spec (list): List of feature columns used in the model.
        outcome_var (str): Target variable name.
        align_year (int): Year for alignment/prediction.
        model_coef (pd.DataFrame): DataFrame containing model coefficients for error estimation.

    Returns:
        pd.DataFrame: DataFrame with columns ['iso3', 'year', 'outcome_var', 'model_type', 'pred', 'true', 'predicted_error'].
    """
    model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl'
    model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
    model = load_model(model_filepath)
    if model is None:
        raise ValueError("Model not found. Ensure the model has been trained and scaler saved.")

    predicted_results = pd.DataFrame(columns=["iso3", "year", 'outcome_var', 'model_type', 'pred', 'true', 'predicted_error'])
    new_data = new_data.dropna(subset=model_spec)
    new_data = new_data.loc[~new_data['iso3'].isin(params.sanction_countries)]
    X_new = preprocess_data(new_data[model_spec])
    X_new['const'] = 1
    X_new = X_new[['const'] + model_spec]
    y_pred_full = model.predict(X_new)

    predicted_results['iso3'] = new_data['iso3']
    predicted_results['outcome_var'] = outcome_var
    predicted_results['model_type'] = model_type
    predicted_results['pred'] = y_pred_full
    predicted_results["year"] = align_year
    predicted_results['true'] = [
        y if x == align_year else None
        for x, y in zip(new_data[f'{indicator}_year'], new_data[outcome_var])
    ]
    predicted_results['predicted_error'] = X_new[model_spec] @ model_coef[model_spec].T

    return predicted_results


In [ ]:

predicted_results = pd.DataFrame()
coef= pd.read_csv(params.dropbox_data_path.parent / f'results/ols_predicted_by_year/{model_type}_country_model_error_estimation_betas.csv')
for indicator in ['internet','mobile']:
    for year in range(2015,2026):
        print(indicator,year)
        data_path = params.dropbox_data_path / f"combined_data/updated_ground_truth_and_fb/{indicator}"
        dataset = f"year_align/combined_multiple_years_no_missing_fb_aligned_{year}.csv"
        new_data = pd.read_csv(data_path/dataset)
        new_data['year']=new_data[f'align_year']-2015
       
        for outcome_var in [f'{indicator}_ggi', f'{indicator}_wom', f'{indicator}_men']:
            #for model_type in ['combined', 'combined_top_shaps_no_fertility']:
            model_type = 'combined_with_CIS'
            model_spec = model_specs[model_type]
            model_coef = coef.loc[coef['model']==outcome_var]
            temp = full_model_predictions(indicator, new_data, model_type, model_spec,outcome_var,year,model_coef)
            predicted_results = pd.concat([predicted_results,temp], axis=0)

predicted_results['model_type'] = model_type
predicted_results['pred'] = [1 if x>1 else 0 if x<0 else x for x in predicted_results['pred']]
predicted_results['predicted_error'] = [1 if x>1 else 0 if x<0 else x for x in predicted_results['predicted_error']]

In [ ]:
predicted_results

In [ ]:
import geopandas as gpd


# Download the shapefile from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/
# and extract it to a known location, e.g., './data/naturalearth/ne_110m_admin_0_countries.shp'

world = gpd.read_file(Path.cwd().parent / 'files/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp')
world = world[(world['POP_EST'] > 0) & (world['NAME'] != "Antarctica")]

outcome_var_dict = {'ggi': 'Gender Gap Indicator','wom':'Women Level','men':'Men Level'}
outcome_vars = ['ggi','wom','men']

for indicator in ['internet','mobile']:
    # Create the figure with 6 subplots (3 rows, 2 columns)
    fig, axs = plt.subplots(3, 2, figsize=(15, 16))
    axs = axs.flatten()

    # Define the minimum and maximum values for the color scale
    vmin = min(df_all_res.loc[df_all_res['outcome_var'].str.contains(indicator),'true'].min(), df_all_res.loc[df_all_res['outcome_var'].str.contains(indicator),'pred'].min())
    vmax = max(df_all_res.loc[df_all_res['outcome_var'].str.contains(indicator),'true'].max(), df_all_res.loc[df_all_res['outcome_var'].str.contains(indicator),'pred'].max())

    # Define the colormap
    cmap = 'coolwarm_r'

    # Loop over outcome variables and plot both predicted and true values
    for i, outcome_var in enumerate(outcome_vars):
        # Filter data for current outcome_var
        df_res = df_all_res[df_all_res['outcome_var'] == f'{indicator}_{outcome_var}']

        # Merge predicted values with world map
        world_pred = world.merge(df_res[['iso3', 'pred']], left_on='iso_a3', right_on='iso3', how='left')

        # Plot predicted values on the left side (first column)
        world_pred.boundary.plot(ax=axs[2 * i], linewidth=1)
        world_pred.plot(column='pred', ax=axs[2 * i], cmap=cmap, missing_kwds={"color": "white"},
                        vmin=vmin, vmax=vmax)
        axs[2 * i].set_title(f'{outcome_var_dict[outcome_var]} - Predicted',fontweight='bold')
        axs[2 * i].axis('off')

        # Merge true values with world map
        world_true = world.merge(df_res[['iso3', 'true']], left_on='iso_a3', right_on='iso3', how='left')

        # Plot true values on the right side (second column)
        world_true.boundary.plot(ax=axs[2 * i + 1], linewidth=1)
        world_true.plot(column='true', ax=axs[2 * i + 1], cmap=cmap, missing_kwds={"color": "white"},
                        vmin=vmin, vmax=vmax)
        axs[2 * i + 1].set_title(f'{outcome_var_dict[outcome_var]} - True',fontweight='bold')
        axs[2 * i + 1].axis('off')

    # Create a single colorbar at the center bottom of the figure
    # Adjust the spacing between subplots

    # Create a single colorbar at the center bottom of the figure
    cbar_ax = fig.add_axes([0.3, 0.05, 0.4, 0.02])  # Position: [left, bottom, width, height]
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # Add the colorbar
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')

    # Adjust layout to ensure padding
    plt.subplots_adjust(wspace=0.1, hspace=0.15)  # Increase horizontal and vertical space between plots
    # fig.suptitle(f'{indicator.capitalize()}  Predicted vs True Values', fontsize=16, fontweight='bold', y=0.95)

    plt.tight_layout(rect=[0, 0.01, 1, 0.99])

    # Show the plot
    # plt.show()
    # plt.savefig(params.dropbox_data_path.parent / f'graphs/{indicator}_pred_vs_true_map.pdf')



# 4. Produce outcomes by month

In [ ]:
current_year = 2025 
current_month = 7
dropbox_fb_path = Path("/Users/valler/Dropbox/dgg_research/pipeline/")
df_fb_data = pd.read_csv(dropbox_fb_path / f'preprocessed/national/fb_national_sd_rolling_std_{current_year}{current_month:02d}.csv')



fb_cols =["fb_18_999_men","fb_18_999_wom","fb_18_999_r"]
#bg_cols = params.bg_cols
bg_cols = ['hdi','gdi','gdp_pcap','year']
model_specs = {#'online': fb_cols, 
           #'offline': bg_cols, 
           #'combined': fb_cols + bg_cols,
           #'combined_top_shaps':['gggi_ggi','se_r','hdi','gdp_pcap','wdi_fertility','year']+fb_cols,
           #'combined_top_shaps_no_fertility':['gggi_ggi','se_r','hdi','gdp_pcap','year']+fb_cols,
           "combined_with_CIS":fb_cols + bg_cols}
           #"combined_top_shaps_from_mobile":['wdi_litrad','gni_pc','hdi','year','ineq_le','wdi_acel','gii']+fb_cols}


In [ ]:


model_folder = '18_plus'
model_type = 'combined_with_CIS'
leave_column = 'country'
model_spec = model_specs[model_type]

fb_result_path = params.dropbox_result_path/f'{model_type}'
for current_year in range(2015, 2026):
    for current_month in range(1,13):
        if (current_month>=2) and (current_year)<2019:
            continue
        else:
            date = f'{current_year}-{str(current_month).zfill(2)}'
            if not os.path.isfile(fb_result_path / f'{date}.csv'):

                fb_cols = params.fb_vars_18_plus
                df_result = pd.DataFrame(columns=['gid_0', 'outcome', 'true', 'predicted', 'date'])
            
            
                for indicator in ['internet', 'mobile']:
                    for outcome_var in [f'{indicator}_ggi',f'{indicator}_wom', f'{indicator}_men']:
    
                        #print(outcome_var)
                        # read file based on year
                        df_model = pd.read_csv(dropbox_fb_path / f'model_fit/national/data/{indicator}/combined_multiple_years_no_missing_fb_aligned_{round(current_year)}.csv')
            
                        if current_year >= 2019:
                            for col in fb_cols:
                                if col in df_model.columns:
                                    df_model.drop(columns=col, inplace=True)
                            df_fb_data_selected = df_fb_data.loc[(df_fb_data['year'] == current_year) & (df_fb_data['month'] == current_month)]
                            df_model = df_model.merge(df_fb_data_selected, left_on=['iso3', 'align_year'], right_on=['iso3', 'year'], how='left')
                        
                        
                        #df_model.drop(columns=[f'{indicator}_year'], inplace=True)
                        
                        df_model['year']=df_model['align_year']-2015
                        df_model = df_model.dropna(subset=model_spec)
            
                            # load the model
                        model_filename = f'{indicator}_{model_type}_{outcome_var}_full_model.pkl' 
                        model_filepath = params.dropbox_data_path.parent / f'models/OLS' / model_filename
                        
                        temp = pd.DataFrame(columns=['gid_0','outcome','true', 'predicted','date'])
                        model = load_model(model_filepath)
            
                        df_model = df_model.groupby(['iso3']).mean(numeric_only=True).reset_index()  # some countries have multiple entries but the results are the same
                        
                        df_model.drop(columns=['align_year'],inplace=True)
                        countries = df_model['iso3'].tolist()
                        df_model['const']=1
                        X = df_model[['const']+model_spec]
                        y_pred = model.predict(X)
            
                        for ind in range(len(y_pred)):
                            true_value = df_model.loc[(df_model[f'{indicator}_year'] == current_year) & (df_model[f'iso3'] == countries[ind]), outcome_var]
                            true_value = true_value.values[0] if len(true_value) > 0 else None
                            temp.loc[len(temp), :] = [countries[ind],outcome_var, true_value, y_pred[ind], date]
            
            
                        # now get the predicted errors
                        df_error_beta = pd.read_csv(params.dropbox_data_path.parent / f'results/ols_predicted_by_year/{model_type}_country_model_error_estimation_betas.csv')
                        coefficients = df_error_beta.loc[(df_error_beta['model'] == outcome_var)].values[0][3:]
            
                        model_spec = df_error_beta.columns.to_list()[3:]
                        df_model.rename(columns={'align_year':'year'}, inplace=True)
                        
                        temp['predicted_error'] = df_model[model_spec] @ coefficients
            
            
                        df_result = pd.concat([df_result, temp], axis=0)
                        #print(df_result['outcome'].unique())
            
                df_result = df_result[['gid_0', 'outcome', 'predicted', 'predicted_error', 'date']]
            
                df_result['outcome'] = df_result['outcome'].str.replace('ggi', 'fm_ratio')
                for col in ['predicted', 'predicted_error']:
                    # only three decimal places
                    df_result[col] = df_result[col].apply(lambda x: round(x, 3))
            
                df_result['outcome'].replace({'mobile_wom': 'mobile_women', 'internet_wom': 'internet_women'}, inplace=True)
                df_result['predicted'] = df_result['predicted'].apply(lambda x: min(x, 1))
                df_result['predicted_error'] = df_result['predicted_error'].apply(lambda x: min(x, 1))
                
                df_result['predicted'] = df_result['predicted'].apply(lambda x: max(x, 0))
                df_result['predicted_error'] = df_result['predicted_error'].apply(lambda x: max(x, 0))
                
                df_result = df_result.loc[~df_result['gid_0'].isin(params.sanction_countries)]

                if not os.path.isfolder(fb_result_path):
                    os.makedirs(fb_result_path, exist_ok=True)
                if current_year>=2019:

                    df_result.to_csv(fb_result_path / f'{date}.csv', index=False)
                else:
                    df_result.to_csv(fb_result_path / f'{round(current_year)}.csv', index=False)

In [ ]:
df = pd.read_csv("/Users/valler/Dropbox/dgg_research/pipeline/result/national/2025-08.csv")


In [ ]:
for outcome in df['outcome'].unique():
    print(outcome, df.loc[df['outcome']==outcome,'gid_0'].nunique())
    print()


In [ ]:

all_countries_from_model = ['ABW', 'AFG', 'AGO', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CUW', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'FRO', 'FSM', 'GAB', 'GBR', 'GEO', 'GHA', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GRL', 'GTM', 'GUM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IMN', 'IND', 'IRL', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KIR', 'KNA', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LCA', 'LIE', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAF', 'MAR', 'MCO', 'MDA', 'MDG', 'MDV', 'MEX', 'MHL', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MNP', 'MOZ', 'MRT', 'MSR', 'MTQ', 'MUS', 'MWI', 'MYS', 'NAM', 'NCL', 'NER', 'NGA', 'NIC', 'NIU', 'NLD', 'NOR', 'NPL', 'NRU', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'PLW', 'PNG', 'POL', 'PRI', 'PRT', 'PRY', 'PSE', 'PYF', 'QAT', 'ROU', 'RWA', 'SAU', 'SEN', 'SGP', 'SHN', 'SLB', 'SLE', 'SLV', 'SMR', 'SOM', 'SRB', 'SSD', 'STP', 'SUR', 'SVK', 'SVN', 'SWE', 'SWZ', 'SXM', 'SYC', 'TCA', 'TCD', 'TGO', 'THA', 'TJK', 'TKM', 'TLS', 'TON', 'TTO', 'TUN', 'TUR', 'TUV', 'TWN', 'TZA', 'UGA', 'UKR', 'URY', 'USA', 'UZB', 'VCT', 'VEN', 'VGB', 'VIR', 'VNM', 'VUT', 'WSM', 'XKX', 'YEM', 'ZAF', 'ZMB', 'ZWE']
print(len(all_countries_from_model))

In [ ]:
set(all_countries_from_model) - set(df.loc[df['outcome']==outcome,'gid_0'].tolist())